# Step 1 — Structural parsing and field separation

**Input:** `equal-data-set.xlsx` (column *Job Advertisement* and column *Source*).
**Output:** five separated fields per advertisement — (a) employer name, (b) job title, (c) qualification block, (d) contact/application instructions, (e) source URL — and a modeling text that excludes the employer, the contact block and URLs while keeping them as metadata.

We adopted a rule-based structural parsing step with our own modifications to the approach of Dikshit et al. (2024). Each advertisement is segmented line by line, every non-empty line is assigned to exactly one field, and the parser records which rule made each assignment. Fields that are not substantive for gender-bias analysis (employer, contact/application instructions, URLs) are removed from the modeling text but retained as metadata so that every record remains traceable to its source row.

## 0. Setup and loading

The workbook is read with `pandas`. The stable `record_id` is the Excel row number (`row-0002` onward) so that every downstream annotation can be traced to the source row. Text is only minimally standardized here (Unicode NFC, line endings, non-breaking spaces, leading bullet glyphs `~ ● ▪ ◦` mapped to `•`) so that the structural rules see a consistent layout; wording, case and punctuation are untouched. Source URLs are read from the *Source* column and are never placed in the modeling text.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

DATA_PATH = Path("equal-data-set.xlsx")
OUTPUT_DIR = Path("outputs") / "equal_step1_structural_parsing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Workbook not found at {DATA_PATH.resolve()}")

raw = pd.read_excel(DATA_PATH, sheet_name=0, dtype=object)
raw.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in raw.columns]
AD_COL, SRC_COL = raw.columns[0], raw.columns[1]


def standardize(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = unicodedata.normalize("NFC", str(text)).replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[\t \u00a0]+", " ", line).strip()
        line = re.sub(r"^[~●▪◦]\s*", "• ", line)
        lines.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


data = pd.DataFrame(
    {
        "record_id": [f"row-{i + 2:04d}" for i in range(len(raw))],
        "standardized_text": raw[AD_COL].map(standardize),
        "source_url": raw[SRC_COL].map(lambda v: "" if pd.isna(v) else str(v).strip()),
    }
)
print(f"{len(data):,} advertisements loaded; {int((data['standardized_text'] == '').sum())} empty texts.")

1,500 advertisements loaded; 0 empty texts.


## 1. Rule definitions

Three kinds of cues are used. They were derived from a frequency count of the header lines and `LABEL:` prefixes actually present in this workbook (e.g. `Qualifications:` appears in 1,292 of 1,500 ads; `For more information, please proceed to:`, `You may send your resume at:` and `EMAIL ADDRESS:` / `MOBILE NO.:` / `TELEPHONE NO.:` lines recur throughout), including observed misspellings (`Qualifiactions`, `Qulifications`, `Ineterested`). The rules are transparent regular expressions, not a trained model, so each decision can be audited.

| Cue | Meaning | Field |
|---|---|---|
| `QUAL_HEADER` | `Qualifications:`, `Qualification Standards:`, `Requirements:`, `Job Description:` (and misspellings) | start of the **qualification block** |
| `CONTACT_HEADER` | a line ending in `:` that introduces application/contact instructions (`send your resume…`, `for more information…`, `interested applicants may…`, `addressed to:`, `head office:`, `requirements for application:` …) | start of the **contact block** |
| `CONTACT_MARKER` | an e-mail, URL, phone-like number, or a labelled contact line (`EMAIL ADDRESS:`, `MOBILE NO.:`, `TELEPHONE NO.:`, `FAX NO.:`, `LINK:`) | **contact** line |
| `APPLY_RE` | a non-bullet sentence giving application instructions without a contact detail (`All interested applicants may proceed to…`, `Send your resume…`) | **contact** line (flagged `apply_instruction_review`) |
| `ADDRESS_CUE` | a non-bullet line with street/barangay/city wording that follows the qualification bullets | **contact** line (flagged as inferred) |

In [2]:
QUAL_HEADER = re.compile(
    r"^(qu?a?l[a-z]*(?:\s+standards)?|requirements?|job\s+description)\s*:\s*(.*)$", re.I
)

CONTACT_HEADER_WORDS = re.compile(
    r"\b(send|resume|resumes|cv|apply|application|applications|applicants?|inquir\w*|"
    r"inquiries|contact|visit|information|e-?mail|mail|addressed|submit|office|address|"
    r"call|recruitment center|place assignment|file your|proceed)\b",
    re.I,
)
CONTACT_HEADER_PREFIX = re.compile(r"^requirements?\s+for\s+application\s*:", re.I)

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w\-]+\.(?:com|ph|org|net|edu|gov)(?:/\S*)?\b", re.I)
PHONE_RE = re.compile(r"(?<!\d)(?:\+?63|0)?[\d][\d\s\-()]{6,}\d(?!\d)")
LABEL_RE = re.compile(
    r"^(?:e-?mail(?:\s+ad+ress)?|email\s+adress|mobile\s+nos?\.?|telephone\s+nos?\.?|"
    r"telepne\s+nos?\.?|tel\.?\s*nos?\.?|contact\s+(?:nos?\.?|numbers?|person)|fax\s+nos?\.?|link|https)\s*:",
    re.I,
)
ADDRESS_CUE = re.compile(
    r"\b(street|st\.|avenue|ave\.?|road|rd\.?|highway|barangay|brgy\.?|purok|zone|"
    r"blvd\.?|boulevard|building|bldg\.?|floor|unit|city|province|municipality|poblacion|pob\.)\b",
    re.I,
)
BULLET_RE = re.compile(r"^•\s*")
APPLY_RE = re.compile(
    r"\b(?:interested|qualified)\b.{0,60}\b(?:applicants?|candidates?)\b"
    r"|\b(?:send|submit|email|forward|bring)\b.{0,40}\b(?:resumes?|cv|application|credentials|documents)\b"
    r"|\b(?:proceed|procedd|walk-?in)\b",
    re.I,
)


def is_contact_header(line):
    if CONTACT_HEADER_PREFIX.match(line):
        return True
    return line.endswith(":") and len(line) <= 110 and bool(CONTACT_HEADER_WORDS.search(line))


def has_contact_marker(line):
    return bool(LABEL_RE.match(line) or EMAIL_RE.search(line) or URL_RE.search(line) or PHONE_RE.search(line))

## 2. The parser, field by field

Parsing is a single top-to-bottom pass with one piece of state (`block` ∈ {body, qualification, contact}). Every non-empty line receives exactly one label and one rule name.

**(a) Employer name → metadata.** In this workbook the advertiser's name is the first non-empty line of the text (e.g. `Yamaha`, `GT168 Mall`, `DMMCIHS`). We take that line as `employer_name` and exclude it from the modeling text, so the model cannot learn employer-specific templates or brand wording. This is a positional rule, so it is a *candidate* that a human should spot-check; the `employer_review` check in Section 4 lists the most frequent values for that purpose. Employer names that reappear inside the body are *not* altered in this step.

**(b) Job title.** The second non-empty line is accepted as `job_title` only if it is not a header, not a contact line, not a bullet and is short (≤ 100 characters). Counts in parentheses such as `Production Associates (70)` (vacancy count) are kept in the raw title but a cleaned `job_title_clean` strips them, so titles are comparable across ads. If no line passes, the title is empty and the record is flagged `no_title`. Titles are kept in the modeling text because gender-marked titles (e.g. *Salesman*) are themselves a bias signal.

**(c) Qualification block.** A `QUAL_HEADER` line opens the block; the lines that follow (normally `•` bullets) belong to it until a contact header, contact marker or inferred address line closes it. Lines before any header that are not contact lines are kept as `body_other` (some ads are one free-text paragraph with no `Qualifications:` header). Both `qualification_text` and `body_other_text` go into the modeling text.

**(d) Contact / application instructions → metadata.** A `CONTACT_HEADER` line opens the contact block, and everything after it stays in the block until a new `QUAL_HEADER` appears. A `CONTACT_MARKER` line seen earlier also opens the block, because contact details are tail material in this corpus. Non-bullet sentences that only give application instructions (`APPLY_RE`) are also moved and flagged `apply_instruction_review`. Non-bullet address lines that follow qualification bullets are assigned to the contact block through `ADDRESS_CUE` and flagged `address_inferred`. A marker found in a long or sentence-style line (e.g. `Kindly send your resume … to x@y.com`) is moved to the contact block as a whole and flagged `inline_contact_review`, because it may also carry non-contact wording that a reviewer should check. E-mails, phone numbers and in-text URLs are additionally extracted into separate metadata columns.

**(e) Source URL → metadata.** The *Source* column is copied unchanged to `source_url`. It never enters the modeling text. A source URL shared by several ads is reported, not de-duplicated (that is a later step).

In [3]:
def parse_advertisement(text):
    lines = [l for l in text.split("\n") if l.strip()]
    out = {
        "employer_name": "",
        "job_title": "",
        "qualification": [],
        "body_other": [],
        "contact": [],
        "flags": set(),
        "rules": [],
    }
    if not lines:
        out["flags"].add("empty_text")
        return out

    out["employer_name"] = re.sub(r"\s*•\s*$", "", lines[0]).strip()
    rest = lines[1:]

    if rest:
        cand = rest[0]
        if (
            len(cand) <= 100
            and not BULLET_RE.match(cand)
            and not QUAL_HEADER.match(cand)
            and not is_contact_header(cand)
            and not has_contact_marker(cand)
        ):
            out["job_title"] = cand
            rest = rest[1:]
    if not out["job_title"]:
        out["flags"].add("no_title")

    block = "body"
    prev_was_bullet = False
    for line in rest:
        m = QUAL_HEADER.match(line)
        if m:
            block = "qualification"
            out["rules"].append("QUAL_HEADER")
            if m.group(2).strip():
                out["qualification"].append(m.group(2).strip())
            prev_was_bullet = False
            continue
        if is_contact_header(line):
            block = "contact"
            out["contact"].append(line)
            out["rules"].append("CONTACT_HEADER")
            prev_was_bullet = False
            continue
        if has_contact_marker(line):
            if block != "contact" and (len(line) > 100 or not LABEL_RE.match(line)) and not (
                EMAIL_RE.fullmatch(line) or URL_RE.fullmatch(line)
            ):
                out["flags"].add("inline_contact_review")
            block = "contact"
            out["contact"].append(line)
            out["rules"].append("CONTACT_MARKER")
            prev_was_bullet = False
            continue
        if not BULLET_RE.match(line) and APPLY_RE.search(line):
            block = "contact"
            out["contact"].append(line)
            out["flags"].add("apply_instruction_review")
            out["rules"].append("APPLY_INSTRUCTION")
            prev_was_bullet = False
            continue
        if (
            block == "qualification"
            and prev_was_bullet
            and not BULLET_RE.match(line)
            and ADDRESS_CUE.search(line)
        ):
            block = "contact"
            out["contact"].append(line)
            out["flags"].add("address_inferred")
            out["rules"].append("ADDRESS_CUE")
            prev_was_bullet = False
            continue
        out["qualification" if block == "qualification" else block if block == "contact" else "body_other"].append(line)
        prev_was_bullet = bool(BULLET_RE.match(line))

    if not out["qualification"]:
        out["flags"].add("no_qualification_block")
    if not out["contact"]:
        out["flags"].add("no_contact_block")
    return out


def clean_title(title):
    return re.sub(r"\s*\(\s*\d+\s*\)\s*$", "", title).strip()


parsed = data["standardized_text"].map(parse_advertisement)

data["employer_name"] = parsed.map(lambda p: p["employer_name"])
data["job_title"] = parsed.map(lambda p: p["job_title"])
data["job_title_clean"] = data["job_title"].map(clean_title)
data["qualification_text"] = parsed.map(lambda p: "\n".join(p["qualification"]))
data["body_other_text"] = parsed.map(lambda p: "\n".join(p["body_other"]))
data["contact_text"] = parsed.map(lambda p: "\n".join(p["contact"]))
data["emails"] = data["contact_text"].map(lambda t: " | ".join(dict.fromkeys(EMAIL_RE.findall(t))))
data["urls_in_text"] = data["contact_text"].map(
    lambda t: " | ".join(dict.fromkeys(u for u in URL_RE.findall(t) if "@" not in u))
)
data["flags"] = parsed.map(lambda p: ";".join(sorted(p["flags"])))
data["rules_fired"] = parsed.map(lambda p: ";".join(dict.fromkeys(p["rules"])))

# The modeling text keeps title, qualification block and any other body text.
data["modeling_text"] = data.apply(
    lambda r: "\n".join(x for x in [r["job_title_clean"], r["body_other_text"], r["qualification_text"]] if x),
    axis=1,
)
print("Parsing complete.")

Parsing complete.


## 3. Validation of the separation

We check (i) **losslessness** — every non-empty line of the standardized text is accounted for exactly once as employer, title, qualification, body or contact (cleaned vacancy counts and moved `Qualifications:`-style header lines are the only intended differences); (ii) **no leakage** — no e-mail, URL or phone-like string remains in the modeling text; and (iii) **field coverage**, i.e. how many ads yielded each field. These checks do not prove the labels are *correct*; they show the separation is complete and consistent. Correctness is assessed by the manual spot-check in Section 5.

In [4]:
def n_lines(t):
    return len([l for l in t.split("\n") if l.strip()])


qual_header_lines = parsed.map(lambda p: p["rules"].count("QUAL_HEADER"))
accounted = (
    (data["employer_name"] != "").astype(int)
    + (data["job_title"] != "").astype(int)
    + data["qualification_text"].map(n_lines)
    + data["body_other_text"].map(n_lines)
    + data["contact_text"].map(n_lines)
    + qual_header_lines
)
total = data["standardized_text"].map(n_lines)
lossless = accounted == total
print(f"Lossless line accounting: {int(lossless.sum()):,} of {len(data):,} ads")
if (~lossless).any():
    display(data.loc[~lossless, ["record_id"]].assign(total=total[~lossless], accounted=accounted[~lossless]).head(10))

leak = data["modeling_text"].map(has_contact_marker)
print(f"Ads with contact-like strings left in modeling_text: {int(leak.sum()):,}")

coverage = pd.DataFrame(
    {
        "field": ["employer_name", "job_title", "qualification_text", "body_other_text", "contact_text", "source_url"],
        "ads_with_field": [
            int((data[c] != "").sum())
            for c in ["employer_name", "job_title", "qualification_text", "body_other_text", "contact_text", "source_url"]
        ],
    }
)
coverage["share"] = (coverage["ads_with_field"] / len(data)).round(3)
display(coverage)

flag_counts = data["flags"].str.split(";").explode().replace("", pd.NA).dropna().value_counts()
print("Review flags:")
display(flag_counts.to_frame("ads"))
print("Rules fired (ads):")
display(data["rules_fired"].str.split(";").explode().replace("", pd.NA).dropna().value_counts().to_frame("lines_or_ads"))

Lossless line accounting: 1,500 of 1,500 ads
Ads with contact-like strings left in modeling_text: 0


,field,ads_with_field,share
0,employer_name,1500,1.000
1,job_title,1497,0.998
2,qualification_text,1343,0.895
3,body_other_text,23,0.015
4,contact_text,1218,0.812
5,source_url,1499,0.999


Review flags:


,ads
flags,
address_inferred,712
no_contact_block,282
inline_contact_review,170
no_qualification_block,157
apply_instruction_review,138
no_title,3


Rules fired (ads):


,lines_or_ads
rules_fired,
QUAL_HEADER,1344
CONTACT_MARKER,1160
ADDRESS_CUE,712
CONTACT_HEADER,230
APPLY_INSTRUCTION,138


## 4. Metadata checks: employer and source URL

The most frequent `employer_name` values and the most repeated `source_url` values are listed so a reviewer can confirm that the first-line rule really captured advertiser names and can see how source links are shared. Nothing is merged or removed here.

In [5]:
print("Most frequent employer_name values (first-line rule):")
display(data["employer_name"].value_counts().head(15).to_frame("ads"))
print("Most repeated source_url values:")
display(data.loc[data["source_url"] != "", "source_url"].value_counts().head(5).to_frame("ads"))
print(f"Unique source URLs: {data.loc[data['source_url'] != '', 'source_url'].nunique():,}")
print(f"Ads without a source URL: {int((data['source_url'] == '').sum())}")

Most frequent employer_name values (first-line rule):


,ads
employer_name,
AsiaPeopleWorks! Inc.,45
Sta. Clara International Corp.,34
"Delta Earthmoving, Inc.",25
Vicente T. Lao Construction,25
Rosabaya Distributors Incorporated,24
"Techno-Trade Resources, Inc.",24
"United Wil Manpower and Allied Services, Inc.",23
"ICAREERUP, Inc.",21
Santos Land Development Corporation,20


Most repeated source_url values:


,ads
source_url,
https://web.facebook.com/photo/?fbid=2506074709616045&set=pcb.2506076996282483,57
https://web.facebook.com/photo?fbid=2399932480230269&set=pcb.2399932623563588,20
https://web.facebook.com/photo/?fbid=2399932486896935&set=pcb.2399932623563588,14
https://web.facebook.com/photo/?fbid=2910035105886668&set=pcb.2910035505886628,13
https://web.facebook.com/photo/?fbid=2910035099220002&set=pcb.2910035505886628,12


Unique source URLs: 1,082
Ads without a source URL: 1


## 5. Side-by-side examples (manual spot-check)

A reproducible sample (seed 42) of ads from each structural pattern is shown with the five fields separated. The contact block in the examples contains real contact details, so treat saved outputs of this cell as restricted.

In [6]:
def show(row):
    print("\n" + "=" * 90)
    print(f"{row['record_id']}   flags: {row['flags'] or '-'}   rules: {row['rules_fired'] or '-'}")
    print(f"(a) employer_name    : {row['employer_name']}")
    print(f"(b) job_title        : {row['job_title']}   ->  clean: {row['job_title_clean']}")
    print("(c) qualification    :\n" + (row["qualification_text"][:500] or "   -"))
    if row["body_other_text"]:
        print("    body_other       :\n" + row["body_other_text"][:400])
    print("(d) contact (metadata):\n" + (row["contact_text"][:400] or "   -"))
    print(f"(e) source_url       : {row['source_url']}")
    print("--> modeling_text    :\n" + row["modeling_text"][:600])


patterns = {
    "standard: Qualifications + contact": data[(data["qualification_text"] != "") & (data["contact_text"] != "") & (data["flags"] == "")],
    "no qualification header": data[data["flags"].str.contains("no_qualification_block")],
    "inline contact sentence": data[data["flags"].str.contains("inline_contact_review")],
    "application instruction sentence": data[data["flags"].str.contains("apply_instruction_review")],
    "address inferred": data[data["flags"].str.contains("address_inferred")],
}
for name, subset in patterns.items():
    print(f"\n##### {name}: {len(subset):,} ads")
    if len(subset):
        show(subset.sample(n=1, random_state=RANDOM_STATE).iloc[0])


##### standard: Qualifications + contact: 125 ads

row-0038   flags: -   rules: QUAL_HEADER;CONTACT_HEADER;CONTACT_MARKER
(a) employer_name    : Puregold
(b) job_title        : Cashier (1)   ->  clean: Cashier
(c) qualification    :
• Female
• Graduate of any 4-year course or at leasr 2 years in college
• With pleasing personality
• With or without experience
(d) contact (metadata):
For inquiries and referral, visit or contact:
• web.facebook.com/LMDPESOMarinduqueOfficial
• lmdpeso@yahoo.com
• Tel. No.: (042) 332-0266
• Mobile No.: 0912-673-0915
• Capitol Compound, Santol, Boac, Marinduque
(e) source_url       : https://web.facebook.com/LMDPESOMarinduqueOfficial/photos/pcb.1191833051248090/1191832531248142/
--> modeling_text    :
Cashier
• Female
• Graduate of any 4-year course or at leasr 2 years in college
• With pleasing personality
• With or without experience

##### no qualification header: 157 ads

row-0285   flags: apply_instruction_review;no_qualification_block   rules: APPLY_

## 6. Exports

The two outputs are split by sensitivity. `step1_modeling_fields.csv` holds only what may enter later modeling steps (title, qualification, body, modeling text, flags). `step1_restricted_metadata.csv` holds the employer, contact block, extracted e-mails/URLs and the source URL, and should be stored with restricted access. Both are keyed by `record_id`.

In [7]:
data[["record_id", "job_title", "job_title_clean", "qualification_text", "body_other_text", "modeling_text", "flags"]].to_csv(
    OUTPUT_DIR / "step1_modeling_fields.csv", index=False, encoding="utf-8-sig"
)
data[["record_id", "employer_name", "contact_text", "emails", "urls_in_text", "source_url", "rules_fired"]].to_csv(
    OUTPUT_DIR / "step1_restricted_metadata.csv", index=False, encoding="utf-8-sig"
)
print("Saved:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n  ")

Saved:
  step1_modeling_fields.csv
  step1_restricted_metadata.csv


## 7. Limitations

- The rules are tuned to the layouts and misspellings seen in this workbook; a new data source needs the header list re-derived.
- The employer is a positional candidate (first line) and may be wrong when an ad starts with something else; employer names repeated in the body are not removed here.
- Address lines are only recognized when they appear after qualification bullets or inside a contact block; a bare address elsewhere stays in the body and is surfaced for review rather than silently removed.
- Inline contact sentences are moved as a whole line, which can drop a small amount of substantive wording; they are flagged `inline_contact_review` for manual checking.
- No labels, clusters or bias scores are produced in this step.